In [1]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2


URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)


In [2]:
def parse_gtfs_feed(response):
    """
    Parse a GTFS-RT response into a FeedMessage.

    Parameters
    ----------
    response : requests.Response
        HTTP response containing the serialized GTFS-RT feed.

    Returns
    -------
    FeedMessage
        Parsed GTFS-RT feed.
    """
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print(f"Number of entities: {len(feed.entity)}")

    return feed

feed = parse_gtfs_feed(response)


Number of entities: 21698


In [3]:
for entity in feed.entity[:10]:
    print(entity)

id: "617693tu"
trip_update {
  trip {
    trip_id: "617693"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788568230
    }
    stop_id: "542513"
    schedule_relationship: SCHEDULED
  }
}

id: "1242215tu"
trip_update {
  trip {
    trip_id: "1242215"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788565320
    }
    stop_id: "374998"
    schedule_relationship: SCHEDULED
  }
}

id: "337776tu"
trip_update {
  trip {
    trip_id: "337776"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    stop_id: "309246"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 1
    stop_id: "278502"
    schedule_relationship: SKIPPED
  }
  stop_time_update {
    stop_sequence: 2
    stop_id: "565180"
    sche

In [4]:
stops_df = pd.read_csv("../data/mvv_stops.csv", delimiter=";")


In [5]:
def create_stop_name_mapping(stops_df):
    """
    Create a mapping from MVV stop IDs to stop names.

    Parameters
    ----------
    stops_df : pandas.DataFrame
        DataFrame containing the MVV stop data. It must contain
        the columns "HstNummer" and "Name ohne Ort".

    Returns
    -------
    dict
        Dictionary mapping stop IDs to stop names.
    """
    stops_df["HstNummer"] = stops_df["HstNummer"].astype(str)

    stop_names = (
        stops_df
        .set_index("HstNummer")["Name ohne Ort"]
        .to_dict()
    )

    return stop_names


In [6]:
def parse_trip_updates(feed, stop_names, trip_lines):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.

    Parameters
    ----------
    feed : FeedMessage
        Parsed GTFS-RT feed containing trip updates.
    stop_names : dict
        Mapping from stop IDs to stop names.
    trip_lines : dict
        Mapping from trip IDs to line names.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing trip, line, stop, arrival,
        departure, and delay information.
    """
    rows = []

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        # Get line for this trip
        line = trip_lines.get(str(trip.trip_id))

        # Ignore trips that are not part of the selected agencies
        if line is None:
            continue

        for stop in entity.trip_update.stop_time_update:

            row = {
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "line": line,
                "stop_id": str(stop.stop_id),
                "stop_name": stop_names.get(str(stop.stop_id)),
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("departure"):
                row["departure_time"] = datetime.fromtimestamp(
                    stop.departure.time
                )
                row["departure_delay"] = stop.departure.delay

            if stop.HasField("arrival"):
                row["arrival_time"] = datetime.fromtimestamp(
                    stop.arrival.time
                )
                row["arrival_delay"] = stop.arrival.delay

            rows.append(row)

    return pd.DataFrame(rows)

In [7]:
def preprocess_gtfs(data_dir, munich_agencies):
    """
    Preprocess GTFS static data for selected agencies.

    Returns
    -------
    trip_lines : dict
        Mapping from trip_id to line name.

    stop_names : dict
        Mapping from stop_id to stop name.
    """

    routes_df = pd.read_csv(f"{data_dir}/routes.txt")
    trips_df = pd.read_csv(f"{data_dir}/trips.txt")
    stops_df = pd.read_csv(f"{data_dir}/stops.txt")

    routes_df["route_id"] = routes_df["route_id"].astype(str)
    routes_df["agency_id"] = routes_df["agency_id"].astype(str)

    trips_df["trip_id"] = trips_df["trip_id"].astype(str)
    trips_df["route_id"] = trips_df["route_id"].astype(str)

    stops_df["stop_id"] = stops_df["stop_id"].astype(str)

    munich_routes = routes_df[
        routes_df["agency_id"].isin(munich_agencies)
    ]

    route_lines = (
        munich_routes
        .set_index("route_id")["route_short_name"]
        .to_dict()
    )

    munich_trips = trips_df[
        trips_df["route_id"].isin(route_lines)
    ]

    trip_lines = (
        munich_trips
        .set_index("trip_id")["route_id"]
        .map(route_lines)
        .to_dict()
    )

    stop_names = (
        stops_df
        .set_index("stop_id")["stop_name"]
        .to_dict()
    )

    return trip_lines, stop_names


In [8]:
munich_agencies = ["100", "191", "364"]

trip_lines, stop_names = preprocess_gtfs(
    "../data",
    munich_agencies
)

In [9]:
df = parse_trip_updates(
    feed,
    stop_names,
    trip_lines
)

df.head(100)

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,260428,20260905,N27,158413,Karlsplatz (Stachus),18,2026-09-05 01:35:00,0.0,2026-09-05 01:35:00,300.0
1,260428,20260905,N27,682626,Ottostraße,19,2026-09-05 01:36:30,0.0,2026-09-05 01:36:30,0.0
2,525380,20260904,U6,92155,Implerstraße,0,2026-09-05 00:35:16,16.0,NaT,NaN
3,525380,20260904,U6,520586,Poccistraße,1,2026-09-05 00:39:15,-105.0,2026-09-05 00:38:53,-127.0
4,525380,20260904,U6,283525,Goetheplatz,2,2026-09-05 00:40:44,-76.0,2026-09-05 00:40:25,-95.0
...,...,...,...,...,...,...,...,...,...,...
95,1242239,20260905,N45,360529,Humboldtstraße,9,2026-09-05 01:50:30,0.0,2026-09-05 01:50:30,0.0
96,1242239,20260905,N45,657847,Claude-Lorrain-Straße,10,2026-09-05 01:51:30,0.0,2026-09-05 01:51:30,0.0
97,1242239,20260905,N45,433788,Baldeplatz,11,2026-09-05 01:53:00,0.0,2026-09-05 01:53:00,0.0
98,1242239,20260905,N45,400722,Kapuzinerstraße,12,2026-09-05 01:54:00,0.0,2026-09-05 01:54:00,0.0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7091 entries, 0 to 7090
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   trip_id          7091 non-null   object        
 1   start_date       7091 non-null   object        
 2   line             7091 non-null   object        
 3   stop_id          7091 non-null   object        
 4   stop_name        7091 non-null   object        
 5   stop_sequence    7091 non-null   int64         
 6   departure_time   6502 non-null   datetime64[ns]
 7   departure_delay  6502 non-null   float64       
 8   arrival_time     6310 non-null   datetime64[ns]
 9   arrival_delay    6310 non-null   float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(5)
memory usage: 554.1+ KB
